# Commentary Feature — EDA + Training + Ablation Comparison

Three sections, designed for repeated iteration as commentary coverage grows over time:

1. **EDA** — sentiment distributions, per-rider trajectories, coverage gaps, a sanity-check correlation between `commentary_form_5` and actual finishing tier
2. **Training** — trains baseline (no commentary) and enriched (with commentary) models with identical hyperparameters/split, saving both to `models/ablation_baseline/` and `models/ablation_enriched/`
3. **Ablation comparison** — corrected per-model-family comparison (holds architecture fixed so the delta isolates the feature's effect)

**Iterative workflow**: after pulling more transcripts and running extraction (`python scripts/run_commentary.py --extract`), re-run `python scripts/run_ingestion.py --prod --force-features` to rebuild `model_df.csv` with updated coverage, then re-run Sections 1–3 below top to bottom to get fresh numbers.

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from peloton_iq.config import MODELS_DIR, MODEL_DF_PATH, settings
from peloton_iq.commentary.form_features import build_sentiment_table, coverage_report
from peloton_iq.prediction.trainer import train, load_model_df

pd.set_option('display.max_columns', 50)
plt.rcParams['figure.figsize'] = (10, 5)

COMMENTARY_COLS = ['commentary_form_5', 'commentary_form_12mo', 'commentary_obs_count']

## 1. EDA — Commentary Sentiment Data

### 1.1 Build the sentiment table and overall coverage

In [ ]:
sentiment_df = build_sentiment_table()
report = coverage_report(sentiment_df)

print(f"Total observations : {report['total_observations']}")
print(f"Unique riders       : {report['unique_riders']}")
print(f"Date range          : {report['date_range']}")
print(f"Riders with 2+ obs  : {report['riders_with_2plus_obs']}")
print(f"Riders with 5+ obs  : {report['riders_with_5plus_obs']}")
sentiment_df.head(10)

### 1.2 Sentiment label distribution — overall and by year

In [ ]:
sentiment_df['year'] = sentiment_df['Date'].dt.year

label_map = {1.0: 'positive', 0.0: 'neutral', -1.0: 'negative'}
sentiment_df['label'] = sentiment_df['sentiment_score'].map(label_map)

overall_counts = sentiment_df['label'].value_counts()
print('Overall distribution:')
print(overall_counts)
print()
print(f"Positive ratio: {overall_counts.get('positive', 0) / len(sentiment_df):.1%}")
print(f"Negative ratio: {overall_counts.get('negative', 0) / len(sentiment_df):.1%}")

In [ ]:
by_year = sentiment_df.groupby(['year', 'label']).size().unstack(fill_value=0)
by_year_pct = by_year.div(by_year.sum(axis=1), axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
by_year.plot(kind='bar', stacked=True, ax=axes[0], color=['#d62728', '#7f7f7f', '#2ca02c'])
axes[0].set_title('Sentiment observation count by year')
axes[0].set_ylabel('Count')

by_year_pct.plot(kind='bar', stacked=True, ax=axes[1], color=['#d62728', '#7f7f7f', '#2ca02c'])
axes[1].set_title('Sentiment proportion by year')
axes[1].set_ylabel('Proportion')
plt.tight_layout()
plt.show()

### 1.3 Coverage gaps — which years/eras are thinnest?

Directly useful for deciding where pulling more transcripts would help most.

In [ ]:
obs_per_year = sentiment_df.groupby('year').size()
races_per_year = sentiment_df.groupby('year')['race_label'].nunique()

coverage_by_year = pd.DataFrame({
    'observations': obs_per_year,
    'unique_races': races_per_year,
})
coverage_by_year['obs_per_race'] = coverage_by_year['observations'] / coverage_by_year['unique_races']
coverage_by_year

### 1.4 Top covered riders and their sentiment trajectory

In [ ]:
top_riders = sentiment_df.groupby('Name').size().sort_values(ascending=False).head(8).index.tolist()
print('Plotting:', top_riders)

fig, ax = plt.subplots(figsize=(14, 6))
for rider in top_riders:
    rider_df = sentiment_df[sentiment_df['Name'] == rider].sort_values('Date')
    rider_df = rider_df.copy()
    rider_df['rolling_sentiment'] = rider_df['sentiment_score'].rolling(5, min_periods=2).mean()
    ax.plot(rider_df['Date'], rider_df['rolling_sentiment'], marker='o', markersize=3, label=rider, alpha=0.8)

ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_title('Rolling 5-observation sentiment score over time, top covered riders')
ax.set_ylabel('Sentiment score (-1 to +1)')
ax.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0))
plt.tight_layout()
plt.show()

### 1.5 Sanity check — does `commentary_form_5` correlate with real outcomes?

Independent of any model: does the lagged sentiment feature actually relate to the rider's real finishing tier? If this shows no relationship, the ablation result downstream is suspect regardless of what the model says.

In [ ]:
model_df = load_model_df(MODEL_DF_PATH)

has_commentary = model_df['commentary_obs_count'] > 0
print(f"Rows with any prior commentary: {has_commentary.sum():,} / {len(model_df):,} ({has_commentary.mean():.1%})")

covered = model_df[has_commentary].copy()
covered.groupby('tier')['commentary_form_5'].agg(['mean', 'median', 'count']).reindex(
    ['winner', 'podium', 'top10', 'top20', 'finisher', 'dnf']
)

In [ ]:
tier_order = ['winner', 'podium', 'top10', 'top20', 'finisher', 'dnf']
fig, ax = plt.subplots(figsize=(10, 5))
covered.boxplot(column='commentary_form_5', by='tier', ax=ax,
                positions=range(len(tier_order)))
ax.set_xticklabels(tier_order)
ax.set_title('commentary_form_5 distribution by actual finishing tier\n(rows with commentary coverage only)')
ax.set_ylabel('commentary_form_5 (lagged sentiment score)')
plt.suptitle('')
plt.tight_layout()
plt.show()

## 2. Training — Baseline vs. Enriched

Trains two full models (identical hyperparameter search, identical temporal train/test split — only the feature set differs) and saves both to disk. **This is the slow part** — at 50 Optuna trials per model family, expect roughly the same runtime as your last run (historically ~3 hours total for both models). Drop `--trials` lower for a faster sanity-check pass before committing to a full run.

Re-run this section every time you've pulled more commentary and rebuilt `model_df.csv` (`python scripts/run_ingestion.py --prod --force-features`).

In [ ]:
N_TRIALS = 50  # lower this (e.g. 15) for a fast sanity-check pass

df = load_model_df(MODEL_DF_PATH)
missing = [c for c in COMMENTARY_COLS if c not in df.columns]
if missing:
    raise RuntimeError(
        f"model_df.csv is missing commentary columns: {missing}\n"
        f"Run: python scripts/run_ingestion.py --prod --force-features"
    )

coverage_pct = (df['commentary_obs_count'] > 0).mean() * 100
print(f"model_df: {len(df):,} rows; {coverage_pct:.1f}% have 1+ prior commentary observation")

baseline_cols = [c for c in settings.model_feature_cols if c not in COMMENTARY_COLS]
enriched_cols = baseline_cols + COMMENTARY_COLS
print(f"Baseline feature count: {len(baseline_cols)}")
print(f"Enriched feature count: {len(enriched_cols)} (+{len(COMMENTARY_COLS)} commentary)")

In [ ]:
# --- Train baseline (no commentary) ---
print('=' * 60)
print('TRAINING BASELINE (no commentary features)')
print('=' * 60)

baseline_result = train(
    models_dir=MODELS_DIR / 'ablation_baseline',
    n_trials=N_TRIALS,
    feature_cols=baseline_cols,
)

In [ ]:
# --- Train enriched (with lagged commentary features) ---
print('=' * 60)
print('TRAINING ENRICHED (with lagged commentary features)')
print('=' * 60)

enriched_result = train(
    models_dir=MODELS_DIR / 'ablation_enriched',
    n_trials=N_TRIALS,
    feature_cols=enriched_cols,
)

## 3. Ablation Comparison — Baseline vs. Enriched

Same-model-family comparison — holds architecture fixed so the delta isolates the effect of adding commentary features, rather than conflating it with Optuna happening to pick a different model type between the two runs.

This section reads from the `metrics.json` files saved by Section 2, so you can also re-run **just this section** without retraining, any time you want to re-check a comparison you already have artifacts for.

In [ ]:
def load_metrics(run_dir):
    with open(MODELS_DIR / run_dir / 'metrics.json') as f:
        return json.load(f)

baseline_metrics = load_metrics('ablation_baseline')
enriched_metrics = load_metrics('ablation_enriched')

baseline_metrics, enriched_metrics

In [ ]:
def build_per_model_comparison(baseline_metrics, enriched_metrics):
    """
    Same-model-family comparison — holds architecture fixed so the
    delta isolates the effect of adding commentary features, rather
    than conflating it with Optuna picking a different model type.
    """
    rows = []
    for name in baseline_metrics:
        b = baseline_metrics[name]
        e = enriched_metrics[name]
        rows.append({
            'model':              name,
            'baseline_top5':      b['top5_ranking_accuracy'],
            'enriched_top5':      e['top5_ranking_accuracy'],
            'delta_top5':         e['top5_ranking_accuracy'] - b['top5_ranking_accuracy'],
            'baseline_log_loss':  b['log_loss'],
            'enriched_log_loss':  e['log_loss'],
            'delta_log_loss':     e['log_loss'] - b['log_loss'],
            'baseline_auc':       b['auc'],
            'enriched_auc':       e['auc'],
            'delta_auc':          e['auc'] - b['auc'],
        })
    return pd.DataFrame(rows).set_index('model')

comparison_df = build_per_model_comparison(baseline_metrics, enriched_metrics)
comparison_df.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2ca02c' if d > 0 else ('#d62728' if d < 0 else '#7f7f7f') for d in comparison_df['delta_top5']]
ax.bar(comparison_df.index, comparison_df['delta_top5'], color=colors)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Top-5 ranking accuracy delta from adding commentary features\n(same model family, baseline vs enriched)')
ax.set_ylabel('Delta top-5 accuracy')
for i, v in enumerate(comparison_df['delta_top5']):
    ax.text(i, v + (0.001 if v >= 0 else -0.003), f'{v:+.1%}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

### Coverage context for this run

Run alongside the comparison above — tells you whether an improved/degraded result is likely explained by changed coverage since the last iteration.

In [ ]:
report = coverage_report()
print(f"Riders with any commentary  : {report['unique_riders']}")
print(f"Riders with 5+ observations  : {report['riders_with_5plus_obs']}")
print(f"Total sentiment observations : {report['total_observations']}")

model_df = load_model_df(MODEL_DF_PATH)
coverage_pct = (model_df['commentary_obs_count'] > 0).mean() * 100
print(f"\nRow-level coverage in model_df: {coverage_pct:.1f}% of rows have 1+ prior observation")

### Run log — track results across iterations

Append a row each time you re-run Sections 2–3, so you can see the trend as coverage grows. Edit the cell below manually after each run.

In [ ]:
# Manually append a row after each training run to track progress over time
run_log = pd.DataFrame([
    # {'date': '2026-06-22', 'coverage_pct': 16.2, 'lgb_delta_top5': 0.035, 'xgb_delta_top5': 0.000},
])
run_log